In [1]:
!pip install langchain-community pypdf
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install -qU langchain-chroma
!pip install -qU langchain_openai


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import hashlib

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


c:\Users\Técnico em IA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- Padrões estruturais para CCT/documentos jurídicos ---
# CLAUSE_PATTERN  = re.compile(r"(CLÁUSULA\s+\d+[A-Z]?\b[^\n]*)", re.IGNORECASE)
# SECTION_PATTERN = re.compile(r"(§\s*\d+|PARÁGRAFO\s+\w+)", re.IGNORECASE)
# TITLE_PATTERN   = re.compile(r"^\s{0,4}([A-ZÁÉÍÓÚÂÊÎÔÛÃÕÇ\s]{8,})\s*$", re.MULTILINE)


# def enrich_metadata(docs, source_path):
#     """Adiciona clause, section e title ao metadata de cada página."""
#     for doc in docs:
#         text = doc.page_content

#         clause_match = CLAUSE_PATTERN.search(text)
#         doc.metadata["clause"] = clause_match.group(1).strip() if clause_match else "nao_identificada"

#         section_match = SECTION_PATTERN.search(text)
#         doc.metadata["section"] = section_match.group(1).strip() if section_match else "nao_identificada"

#         title_match = TITLE_PATTERN.search(text)
#         doc.metadata["title"] = title_match.group(1).strip() if title_match else "nao_identificado"

#         doc.metadata["source"] = source_path

#     return docs


# def structural_then_char_split(docs, chunk_size=600, chunk_overlap=80):
#     """Split primário por separadores estruturais, depois por caracteres."""
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size=chunk_size,
#         chunk_overlap=chunk_overlap,
#         add_start_index=True,
#         separators=[
#             r"\nCLÁUSULA \d",
#             r"\n§",
#             "\n\n",
#             "\n",
#             " ",
#         ],
#         is_separator_regex=True,
#     )

#     all_chunks = []
#     for doc in docs:
#         chunks = splitter.split_documents([doc])
#         for chunk in chunks:
#             # Refina a cláusula dentro do chunk, se houver uma mais específica
#             detected = CLAUSE_PATTERN.search(chunk.page_content)
#             if detected:
#                 chunk.metadata["clause"] = detected.group(1).strip()
#         all_chunks.extend(chunks)

#     return all_chunks


def deterministic_chunk_id(source, page, chunk_text):
    payload = f"{source}|{page}|{chunk_text.strip().lower()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


In [ ]:
"""Carrega o PDF e enriquece os metadados de cada página."""

file_path = "D:\\UC15\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
# docs = enrich_metadata(docs, file_path)

print(f"Páginas carregadas: {len(docs)}")
print("Exemplo de metadata:", docs[0].metadata)


Páginas carregadas: 38
Exemplo de metadata: {'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-11-24T18:02:52-03:00', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'author': 'Vivianne Morais', 'moddate': '2025-11-24T21:54:55+00:00', 'source': 'D:\\UC15\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'clause': 'nao_identificada', 'section': 'nao_identificada', 'title': ''}


In [ ]:
"""Split the loaded documents into smaller chunks and print the total number of chunks created."""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=80, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

# all_splits = structural_then_char_split(docs, chunk_size=600, chunk_overlap=80)

# print(f"Total de chunks: {len(all_splits)}")
# print("Exemplo de metadata do chunk:", all_splits[0].metadata)

Total de chunks: 225
Exemplo de metadata do chunk: {'producer': 'Microsoft® Word para Microsoft 365 - by Lacuna Software PKI SDK', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-11-24T18:02:52-03:00', 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'author': 'Vivianne Morais', 'moddate': '2025-11-24T21:54:55+00:00', 'source': 'D:\\UC15\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'clause': 'nao_identificada', 'section': 'nao_identificada', 'title': '', 'start_index': 0}


In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# embeddings = HuggingFaceEmbeddings(model_name="embaas/sentence-transformers-multilingual-e5-base")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9788.05it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.0048086754977703094, -0.040888670831918716, -0.023646870627999306, 0.032102070748806, 0.021650686860084534, -0.05819399654865265, 0.010241705924272537, -0.0009653961169533432, -0.052026618272066116, -0.023083817213773727]


In [8]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [9]:
chunk_ids = [
    deterministic_chunk_id(
        chunk.metadata["source"],
        chunk.metadata["page"],
        chunk.page_content
    )
    for chunk in all_splits
]

ids = vector_store.add_documents(documents=all_splits, ids=chunk_ids)

In [10]:
results = vector_store.similarity_search_with_score(
    "Quais as garantias ao retornar de férias?"
)

doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.6680962443351746

page_content='do equivalente às incidências sobre férias integrais e proporcionais  sempre acrescidas do 
terço constitucional, décimo terceiro salário integral e proporcional. 
 
43 - GARANTIA DE EMPREGO - RETORNO DAS FÉRIAS -  O empregado que retornar de 
férias não poderá ser dispensado antes de 30 (trinta) dias, contados  a partir do primeiro dia 
de trabalho. 
 
Parágrafo 1º - Na hipótese do fracionamento das férias individuais e/ou féri as coletivas, a 
garantia de emprego será proporcional aos dias de férias gozados ou abonados. Não serão' metadata={'title': '', 'creator': 'Microsoft® Word para Microsoft 365', 'start_index': 1463, 'documentkey': '5ccd231f-5127-4696-84df-b576074c6eba', 'section': 'Parágrafo único', 'page_label': '21', 'source': 'D:\\UC15\\senac_ia_uc15_nlp_project\\data\\marcelo\\sindilojas_2025_2026.pdf', 'author': 'Vivianne Morais', 'clause': 'cláusula 41.', 'creationdate': '2025-11-24T18:02:52-03:00', 'total_pages': 38, 'moddate': '2